# Footprint-weighted fertilizer rates and grain yield

Combines the aggregated flux footprint climatologies (see `../scripts/`) with the
as-applied fertilization map and the wheat grain yield map to derive, for each
treatment parcel:

* the **footprint-weighted** mean applied N rate and grain yield (what the eddy
  covariance system actually "sees"), and
* the **arithmetic** mean over the whole parcel, for comparison.

**Input** `../data/processed/footprint/kljun_climatology/`, `../data/raw/GIS/`
**Output** printed summary statistics reported in the manuscript


# IMPORTS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
from shapely.affinity import rotate
from rasterio.features import rasterize
from affine import Affine
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import glob
import os

# CONFIG

Station coordinates

In [ ]:
EC_X, EC_Y = 2711032 , 1259910  # set your coordinates
EPSG = 2056 # set UTM EPSG
az_start = 84.6 # sector dividing the treatements in deg (from north, clockwise)
az_end = az_start + 180
# For treatment division line
th0 = np.radians(90.0 - az_start)
th1 = np.radians(90.0 - az_end)

# LOAD DATA

## footprint climatology

### load & process

In [ ]:
# Footprints after each fertilization event in the precision treatment
fc_list = ["Fert1", "Fert2", "Fert3", "Wheat"]
data_dict = {}

for file in fc_list:
    # match files for this fert
    fclim_files = sorted(glob.glob(f"../data/processed/footprint/kljun_climatology/fclim_2d_{file}*_ALL*.csv"))
    x_files     = sorted(glob.glob(f"../data/processed/footprint/kljun_climatology/x_2d_{file}*_ALL*.csv"))
    y_files     = sorted(glob.glob(f"../data/processed/footprint/kljun_climatology/y_2d_{file}*_ALL*.csv"))

    # read the first file for each pattern
    fclim_2d = pd.read_csv(fclim_files[0], header=None).apply(pd.to_numeric, errors="coerce").values[1:, 1:]
    x_2d     = pd.read_csv(x_files[0],     header=None).apply(pd.to_numeric, errors="coerce").values[1:, 1:]
    y_2d     = pd.read_csv(y_files[0],     header=None).apply(pd.to_numeric, errors="coerce").values[1:, 1:]

    # align footprint grid to map coords
    x_centers = x_2d[0, :] + EC_X
    y_centers = y_2d[:, 0] + EC_Y

    fc = fclim_2d.copy(); xc = x_centers.copy(); yc = y_centers.copy()
    if xc[1] < xc[0]:
        fc = fc[:, ::-1]; xc = xc[::-1]
    if yc[1] < yc[0]:
        fc = fc[::-1, :]; yc = yc[::-1]

    nrows, ncols = fc.shape
    dx = float(np.abs(np.diff(xc)).mean())
    dy = float(np.abs(np.diff(yc)).mean())
    x_min, x_max = float(xc.min()), float(xc.max())
    y_min, y_max = float(yc.min()), float(yc.max())
    transform = Affine(dx, 0, x_min - dx/2, 0,  dy, y_min - dy/2)
    extent = (x_min - dx/2, x_max + dx/2, y_min - dy/2, y_max + dy/2)
    # store in dict
    data_dict[file] = {"fclim": fc, 
                       "x": xc, 
                       "y": yc,
                       "dx": dx, 
                       "dy": dy,
                       "extent": extent,
                       "transform": transform
    }
    print(f"Loaded {file} with shape {fc.shape} from:\n{fclim_files[0]}\n{x_files[0]}\n{y_files[0]}\n\n")

### plot

In [ ]:
# Limits for plots
distance = 40  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, file in zip(axes, data_dict.keys()):
    d = data_dict[file]
    fc = d["fclim"]
    x_min, x_max, y_min, y_max = d["extent"]

    im = ax.imshow(fc, extent=(x_min, x_max, y_min, y_max), origin='lower')
    ax.scatter([EC_X], [EC_Y], c='red', s=20, marker='x', zorder=3)

    # sector boundary rays (choose R long enough to cross panel)
    R = max(x_max-EC_X, EC_X-x_min, y_max-EC_Y, EC_Y-y_min)
    ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], 'k', lw=2)
    ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], 'k', lw=2)

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_title(f"Footprint — {file}")
    ax.set_xlabel("Easting [m]")
axes[0].set_ylabel("Northing [m]")

# one shared colorbar
cb = fig.colorbar(im, ax=axes, location='right', fraction=0.025, pad=0.02)
cb.set_label("Footprint contribution")
plt.show()

## fertilizer maps

Fertilization prescription maps indicating the fertilizer rate for VRA

In [ ]:
# Load fertilization polygons
fert_path = "../data/raw/GIS/fertilizations/fertilization_as_applied.shp"  # Update with your actual filename
# Map the dictionary keys to their specific columns in the shapefile
fert_map = {
    'Fert1': 'N_APPLY1',
    'Fert2': 'N_APPLY2',
    'Fert3': 'N_APPLY3'
}
# Load the shapefile
gdf_master = gpd.read_file(fert_path)

Make a grid raster out of the polygons

In [ ]:
if gdf_master.crs is None:
    raise ValueError("Shapefile has no CRS; set it before proceeding.")
if gdf_master.crs.to_epsg() != EPSG:
    gdf_master = gdf_master.to_crs(epsg=EPSG)

# Iterate and Rasterize
for fert_key, fert_col in fert_map.items():
    # Retrieve extent/shape info for this specific fertilization key
    d = data_dict[fert_key]
    x_min, x_max, y_min, y_max = d["extent"]
    nrows, ncols = d["fclim"].shape
    transform = d["transform"]
    # Create bounding box for this specific grid
    bbox_poly = Polygon([(x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max)])
    # Clip the master GDF to this extent
    # Note: We clip the master GDF, but we keep the column we need for this iteration
    gdf_clip = gpd.clip(gdf_master, gpd.GeoSeries([bbox_poly], crs=f"EPSG:{EPSG}"))
    # Prepare shapes using the SPECIFIC column (fert_col) for values
    shapes = [(geom, val) for geom, val in zip(gdf_clip.geometry, gdf_clip[fert_col]) if np.isfinite(val)]
    # Rasterize
    fert_grid = rasterize(
        shapes, 
        out_shape=(nrows, ncols),
        transform=transform, 
        fill=np.nan, 
        dtype="float32"
    )
    # Store results
    data_dict[fert_key]["fert_grid"] = fert_grid
    data_dict[fert_key]["fert_shapefile"] = gdf_clip

Plot the resulting rasters

In [ ]:
# Define the a list of fertilizations 
fert_list = ['Fert1', 'Fert2', 'Fert3']

# Limits for plots
distance = 200  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

im = None
for ax, fert in zip(axes, fert_list):
    d = data_dict[fert]
    fert_grid = d["fert_grid"]
    x_min, x_max, y_min, y_max = d["extent"]
    im = ax.imshow(
        fert_grid, extent=(x_min, x_max, y_min, y_max), origin='lower', cmap='YlGn'
    )
    # tower
    ax.scatter([EC_X], [EC_Y], c='red', s=20, marker='x', zorder=3)
    # sector boundary rays
    R = max(x_max-EC_X, EC_X-x_min, y_max-EC_Y, EC_Y-y_min)
    ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], 'k', lw=2)
    ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], 'k', lw=2)
    # individual colorbar for this panel
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Fertilizer rate [kg/ha]")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_title(f"Fertilization grid — {fert}")
    ax.set_xlabel("Easting [m]")
axes[0].set_ylabel("Northing [m]")

plt.show()


## Yield map

* The map was pre-processed by SFF following a protocol established from HAFL
* I calibrated the yield based on the total yield of the field
* Finally yield DM in tons per hectare was calculated based on the measured moisture when the total yield was weighted
* Polygons close to the station when the harvester was turning were removed to avoid biases due to these operations

In [ ]:
# Settings 
yield_points_path = "../data/raw/GIS/yield/yield_map_wheat_2024.shp"
# Define fields we are interested in
yield_field = "t_ha_DM"    # field name with yield data
eff_field = "eff"
# Load
gy = gpd.read_file(yield_points_path)

In [ ]:
if gy.crs is None:
    raise ValueError("Shapefile has no CRS; set it before proceeding.")
if gy.crs.to_epsg() != EPSG:
    gy = gy.to_crs(epsg=EPSG)

# Rasterize both yield and efficiency maps
for field in [yield_field, eff_field]:
    # Retrieve extent/shape info for wheat
    d = data_dict["Wheat"]
    x_min, x_max, y_min, y_max = d["extent"]
    nrows, ncols = d["fclim"].shape
    transform = d["transform"]
    # Create bounding box for this specific grid
    bbox_poly = Polygon([(x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max)])
    # Clip the yield map to this extent
    field_clip = gpd.clip(gy, gpd.GeoSeries([bbox_poly], crs=f"EPSG:{EPSG}"))
    # Prepare shapes using the SPECIFIC column (e.g., yield field) for values
    shapes = [(geom, val) for geom, val in zip(field_clip.geometry, field_clip[field]) if np.isfinite(val)]
    # Rasterize
    field_grid = rasterize(
        shapes, 
        out_shape=(nrows, ncols),
        transform=transform, 
        fill=np.nan, 
        dtype="float32"
    )
    # Store results
    data_dict["Wheat"][f"{field}_grid"] = field_grid
    data_dict["Wheat"][f"{field}_shapefile"] = field_clip

In [ ]:
# Limits for plots
distance = 40  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6), sharex=True, sharey=True)
for ax, field in zip(axes, [yield_field, eff_field]):
    d = data_dict["Wheat"]
    field_grid = d[f"{field}_grid"]
    x_min, x_max, y_min, y_max = d["extent"]
    im = ax.imshow(
        field_grid, extent=(x_min, x_max, y_min, y_max), origin='lower', cmap='YlGn'
    )
    # tower
    ax.scatter([EC_X], [EC_Y], c='red', s=20, marker='x', zorder=3)
    # sector boundary rays
    R = max(x_max-EC_X, EC_X-x_min, y_max-EC_Y, EC_Y-y_min)
    ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], 'k', lw=2)
    ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], 'k', lw=2)
    # individual colorbar for this panel
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(f"{field} value")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_title(f"{field} grid")
    ax.set_xlabel("Easting [m]")
axes[0].set_ylabel("Northing [m]")
plt.show()

## treatment masks

In [ ]:
buffer_angle = 2 # degrees buffer to avoid edge effects

def ang_dist(a, b):
    # minimal circular distance between angles (0..180)
    return np.abs((a - b + 180.0) % 360.0 - 180.0)

for file, d in data_dict.items():
    xc = d["x"]   # 1D x centers (absolute map coords)
    yc = d["y"]   # 1D y centers (absolute map coords)
    fc = d["fclim"]
    # 2D cell-center coordinates
    X, Y = np.meshgrid(xc, yc) # same shape as fc
    # Offsets from tower
    DX = X - EC_X
    DY = Y - EC_Y
    # Bearing from north, clockwise
    bearing = (90.0 - np.degrees(np.arctan2(DY, DX))) % 360.0
    # Sector → B (inside), A (outside)
    if az_start <= az_end:
        sector_mask = (bearing >= az_start) & (bearing <= az_end)
    else:
        sector_mask = (bearing >= az_start) | (bearing <= az_end)
    # cells close to either boundary line
    near_boundary = (ang_dist(bearing, az_start) <= buffer_angle) | (ang_dist(bearing, az_end) <= buffer_angle)
    # keep only "core" areas (exclude the ±buffer_angle° belt)
    parcel_B = sector_mask & (~near_boundary) & np.isfinite(fc)
    parcel_A = (~sector_mask) & (~near_boundary) & np.isfinite(fc)
    # Store in dict
    d["parcel_A"] = parcel_A
    d["parcel_B"] = parcel_B
    d["parcel_buffer"] = near_boundary & np.isfinite(fc)  # optional, for checking
    # quick sanity check
    print(f"{file}: A={parcel_A.sum()}  B={parcel_B.sum()}  buffer={near_boundary.sum()}  total={np.isfinite(fc).sum()}")

data_dict['Wheat'].keys()

Check the parcel masks on wheat grain yield map

In [ ]:
# Limits for plots
distance = 220  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance

# Overlay plot: yield map with parcel masks
fig, ax = plt.subplots(figsize=(8, 8))
d = data_dict["Wheat"]
x_min, x_max, y_min, y_max = d["extent"]
grid = d[f"{yield_field}_grid"]
# Base yield grid
im = ax.imshow(
    grid, extent=(x_min, x_max, y_min, y_max),
    origin="lower", cmap="YlGn"
)
# Masks (only show True cells)
maskA   = np.ma.masked_where(~d["parcel_A"], d["parcel_A"])
maskB   = np.ma.masked_where(~d["parcel_B"], d["parcel_B"])
ax.imshow(maskA.astype(float), extent=(x_min, x_max, y_min, y_max),
          origin="lower", cmap="Reds", vmin=0, vmax=1, alpha=0.5, interpolation="nearest")
ax.imshow(maskB.astype(float), extent=(x_min, x_max, y_min, y_max),
          origin="lower", cmap="Blues", vmin=0, vmax=1, alpha=0.5, interpolation="nearest")
# Tower
ax.scatter([EC_X], [EC_Y], c="red", s=20, marker="x", zorder=5)
# Sector boundary rays
R = max(x_max-EC_X, EC_X-x_min, y_max-EC_Y, EC_Y-y_min)
ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], "k", lw=2)
ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], "k", lw=2)
# Colorbar for yield
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Yield grain [kg/ha]")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
ax.set_title("Yield grid + parcel overlays")
ax.set_xlabel("Easting [m]")
ax.set_ylabel("Northing [m]")

plt.show()

# AGGREGATE FERT RATES

## footprint-weighted average

In [ ]:
results = []
for fert in fert_list:
    d = data_dict[fert]
    dx = d["dx"]
    dy = d["dy"]
    fc = d["fclim"]
    fert_grid = d["fert_grid"]
    parcel_A = d["parcel_A"]
    parcel_B = d["parcel_B"]
    # Weighted means in both parcels
    overlapA = parcel_A & np.isfinite(fert_grid)
    overlapB = parcel_B & np.isfinite(fert_grid)
    # normalize fc once (so weights sum ~1 over domain)
    total = np.nansum(fc) * dx * dy
    fcn = fc if (np.isfinite(total) and abs(total - 1.0) < 1e-3) else fc / (np.nansum(fc) * dx * dy)
    w = fcn * (dx * dy)  # base weights
    denA = np.nansum(w[overlapA]) # normalization denominator
    denB = np.nansum(w[overlapB])
    meanA = float(np.nansum(fert_grid[overlapA] * w[overlapA]) / denA) if denA > 0 else np.nan
    meanB = float(np.nansum(fert_grid[overlapB] * w[overlapB]) / denB) if denB > 0 else np.nan
    variance_A = np.nansum(w[overlapA] * (fert_grid[overlapA] - meanA)**2) / denA
    stdA = np.sqrt(variance_A)
    variance_B = np.nansum(w[overlapB] * (fert_grid[overlapB] - meanB)**2) / denB
    stdB = np.sqrt(variance_B)
    
    results.append((fert, meanA, meanB, stdA, stdB))

# Print results
print("Applied N rate (kg/ha) with standard deviation based on footprint weights:")
for name, meanA, meanB, stdA, stdB in results:
    print('-'*60)
    print(f"{name}")
    print(f"Parcel A: {meanA:.2f} ± {stdA:.2f} kg/ha")
    print(f"Parcel B: {meanB:.2f} ± {stdB:.2f} kg/ha")

Plot overlay of fc and fert rate

In [ ]:
# Limits for plots
distance = 40  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance

# Figure & axes
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

for ax, fert in zip(axes, fert_list):
    d = data_dict[fert] # d is your dict for this event
    fc = d['fclim']
    extent = d["extent"]
    fert_grid = d["fert_grid"]
    parcel_B = d["parcel_B"]
    # base: fertilization colors
    im = ax.imshow(fert_grid, extent=extent, origin='lower', cmap='YlGn')
    # Overlay alpha from fc restricted to Parcel B
    alpha = np.ones_like(fc, dtype=float)            # default opaque
    valid_B = parcel_B & np.isfinite(fc)
    if valid_B.any():
        vmin, vmax = np.nanmin(fc[valid_B]), np.nanmax(fc[valid_B])
        alpha[valid_B] = 1.0 - (fc[valid_B] - vmin) / (vmax - vmin)  # invert scale
        np.clip(alpha, 0.0, 1.0, out=alpha)
    overlay = np.ones(fc.shape + (4,), dtype=float)
    overlay[..., :3] = 1.0   # white RGB
    overlay[..., 3]  = alpha
    ax.imshow(overlay, extent=extent, origin='lower', interpolation='nearest')
    # Tower + sector lines
    ax.scatter([EC_X], [EC_Y], c='red', s=20, marker='x', zorder=3)
    x_min, x_max, y_min, y_max = extent
    R = max(x_max-EC_X, EC_X-x_min, y_max-EC_Y, EC_Y-y_min)
    ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], 'k', lw=2)
    ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], 'k', lw=2)
    # Cosmetics
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_title(f"Overlay footprint × fertilizer — {fert}")
    ax.set_xlabel("Easting [m]")
    # Per-panel colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.05)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Fertilizer rate [kg/ha]")

axes[0].set_ylabel("Northing [m]")

plt.tight_layout()
plt.show()


## arithmetic average - whole field

In [ ]:
print("Applied N (kg/ha) with standard deviation based on arithmetic average of the whole field:")
for fert in fert_list:
    d = data_dict[fert]
    y = d['fert_grid']
    maskA = d["parcel_A"]
    maskB = d["parcel_B"]
    meanA = np.nanmean(y[maskA])  # ignores NaNs in yield
    meanB = np.nanmean(y[maskB])
    stdevA = np.nanstd(y[maskA])
    stdevB = np.nanstd(y[maskB])
    print('-'*60)
    print(f"{fert}")
    print(f"Parcel A: {meanA:.2f} ± {stdevA:.2f} kg/ha")
    print(f"Parcel B: {meanB:.2f} ± {stdevB:.2f} kg/ha")

# AGGREGATE YIELD

## footprint-weighted average

In [ ]:
for field in [yield_field, eff_field]:
    d = data_dict["Wheat"]
    parcel_A = d["parcel_A"]
    parcel_B = d["parcel_B"]
    fc = d["fclim"]
    grid = d[f"{field}_grid"]
    dx = d["dx"]
    dy = d["dy"]

    # Weighted means in both parcels
    overlapA = (parcel_A) & np.isfinite(fc) & np.isfinite(grid)
    overlapB = (parcel_B) & np.isfinite(fc) & np.isfinite(grid)
    overlap_all = np.isfinite(fc) & np.isfinite(grid)

    # normalize fc once (so weights sum ~1 over domain)
    total = np.nansum(fc) * dx * dy
    fcn = fc if (np.isfinite(total) and abs(total - 1.0) < 1e-3) else fc / (np.nansum(fc) * dx * dy)
    w = fcn * (dx * dy)  # base weights
    denA = np.nansum(w[overlapA])
    denB = np.nansum(w[overlapB])
    den_all = np.nansum(w[overlap_all])
    meanA = float(np.nansum(grid[overlapA] * w[overlapA]) / denA) if denA > 0 else np.nan
    meanB = float(np.nansum(grid[overlapB] * w[overlapB]) / denB) if denB > 0 else np.nan
    mean_all = float(np.nansum(grid[overlap_all] * w[overlap_all]) / den_all) if den_all > 0 else np.nan
    variance_A = np.nansum(w[overlapA] * (grid[overlapA] - meanA)**2) / denA
    stdA = np.sqrt(variance_A)
    variance_B = np.nansum(w[overlapB] * (grid[overlapB] - meanB)**2) / denB
    stdB = np.sqrt(variance_B)
    variance_all = np.nansum(w[overlap_all] * (grid[overlap_all] - mean_all)**2) / den_all if den_all > 0 else np.nan
    std_all = np.sqrt(variance_all)
    if field == yield_field:
        text = "Grain yield (kg/ha)"
    else:
        text = "Efficiency (%)"
    print('-'*60)
    print(f"{text} with standard deviation based on footprint weights:")
    print(f"Parcel A: {meanA:.2f} ± {stdA:.2f}")
    print(f"Parcel B: {meanB:.2f} ± {stdB:.2f}")
    print(f"Overall : {mean_all:.2f} ± {std_all:.2f}")

In [ ]:
# Overlay plot: yield map with footprint alpha
# Assing some missing variables
d = data_dict["Wheat"]
fc = d["fclim"]
extent = d["extent"]
parcel_A = d['parcel_A']
parcel_B = d['parcel_B']
transform = d['transform']
grid = d[f"{yield_field}_grid"]
distance = 50  # m from EC
x0, x1 = EC_X - distance, EC_X + distance
y0, y1 = EC_Y - distance, EC_Y + distance
vmin, vmax = np.nanmin(fc), np.nanmax(fc)
alpha = np.ones_like(fc)
if np.isfinite(vmin) and np.isfinite(vmax) and vmax > vmin:
    alpha = 1.0 - (fc - vmin) / (vmax - vmin)
    np.clip(alpha, 0, 1, out=alpha)

overlay = np.ones(fc.shape + (4,), dtype=float)
overlay[..., :3] = 1.0
overlay[..., 3] = alpha

fig, ax = plt.subplots(figsize=(7,7))
im = ax.imshow(grid, extent=extent, origin="lower", cmap="YlGn")
ax.imshow(overlay, extent=extent, origin="lower")
ax.scatter([EC_X],[EC_Y],c="red",marker="x",s=40,zorder=3)
# sector boundary rays (choose R long enough to cross panel)
ax.plot([EC_X, EC_X + R*np.cos(th0)], [EC_Y, EC_Y + R*np.sin(th0)], 'k', lw=2)
ax.plot([EC_X, EC_X + R*np.cos(th1)], [EC_Y, EC_Y + R*np.sin(th1)], 'k', lw=2)
ax.set_aspect("equal")
ax.set_title(f"Yield × Footprint overlay")
ax.set_xlabel("Easting [m]"); ax.set_ylabel("Northing [m]")
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
# colorbar for yield
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.05)
cb = fig.colorbar(im, cax=cax)
cb.set_label("Grain yield [kg/ha]")
plt.tight_layout()
plt.show()

## arithmetic average - whole field

In [ ]:
for field in [yield_field, eff_field]:
    d = data_dict["Wheat"]
    y = d[f"{field}_grid"]
    maskA = d["parcel_A"]
    maskB = d["parcel_B"]
    meanA = np.nanmean(y[maskA])  # ignores NaNs in yield
    meanB = np.nanmean(y[maskB])
    mean_all = np.nanmean(y[np.isfinite(y)])
    stdevA = np.nanstd(y[maskA])
    stdevB = np.nanstd(y[maskB])
    stdev_all = np.nanstd(y[np.isfinite(y)])
    if field == yield_field:
        text = "Grain yield (kg/ha)"
    else:
        text = "Efficiency (%)"
    print('-'*60)
    print(f"{text} with standard deviation based on arithmetic average of the whole field:")
    print(f"Parcel A: {meanA:.2f} ± {stdevA:.2f}")
    print(f"Parcel B: {meanB:.2f} ± {stdevB:.2f}")
    print(f"Overall : {mean_all:.2f} ± {stdev_all:.2f}")